# DK-SOFNN — Corrected Notebook
This notebook runs the full **Data-Knowledge-Driven Self-Organizing Fuzzy Neural Network**
pipeline by importing from the `DK_SOFNN` module rather than re-implementing everything inline.

Reference: H. Han, X. Liu, and J. Qiao, *IEEE Trans. Neural Netw. Learn. Syst.*, vol. 35, no. 2, Feb. 2024.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from DK_SOFNN import (
    FNN, train_source_fnn, calculate_indexes, source_self_organization,
    adjust_target_structure, train_dk_sofnn, calculate_metrics,
    generate_mackey_glass, print_rule_base,
    SEED, INITIAL_RULES, SOURCE_EPOCHS, SOURCE_LR, TARGET_LR, N_T_DEFAULT,
)
from sklearn.preprocessing import MinMaxScaler

np.random.seed(SEED)

## 1. Generate Data & Split into Source / Target

In [ ]:
X, y = generate_mackey_glass(n_samples=2000, seed=SEED)
feature_names = ['lag1', 'lag2', 'lag3', 'lag4']

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_norm = scaler_X.fit_transform(X)
y_norm = scaler_y.fit_transform(y)

n_source = 1500
n_target_train = 100
n_target_test = 300

X_source,       y_source       = X_norm[:n_source],                          y_norm[:n_source]
X_target_train, y_target_train = X_norm[n_source:n_source+n_target_train],   y_norm[n_source:n_source+n_target_train]
X_target_test,  y_target_test  = X_norm[n_source+n_target_train:n_source+n_target_train+n_target_test], \
                                 y_norm[n_source+n_target_train:n_source+n_target_train+n_target_test]

# Add domain discrepancy noise to target labels
noise = np.random.normal(0, 0.05, y_target_train.shape)
y_target_train_noisy = np.clip(y_target_train + noise, 0, 1)

print(f"Source: {len(X_source)}, Target train: {len(X_target_train)}, Target test: {len(X_target_test)}")

## 2. Train Source FNN & Self-Organize

In [ ]:
n_inputs = X_source.shape[1]
source_fnn = FNN(n_inputs=n_inputs, n_rules=INITIAL_RULES)
source_mse = train_source_fnn(source_fnn, X_source, y_source)

source_fnn = source_self_organization(source_fnn, X_source, y_source, n_iterations=5)

R_s, M_s, C_s = calculate_indexes(source_fnn, X_source, y_source)
source_avgs = (np.mean(R_s), np.mean(M_s), np.mean(C_s))
print(f"Post-self-org averages — R={source_avgs[0]:.4f}, M={source_avgs[1]:.4f}, C={source_avgs[2]:.4f}")

## 3. Source Training MSE Convergence

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(source_mse, linewidth=1.2)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Source FNN Training — MSE Convergence")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Knowledge Transfer & DK-SOFNN Target Training

In [ ]:
best_rule_idx = np.argmax(C_s)
source_expert = {
    'center': source_fnn.centers[best_rule_idx].copy(),
    'width':  source_fnn.widths[best_rule_idx].copy(),
    'weight': source_fnn.weights[best_rule_idx].copy(),
}
source_params = {
    'centers': source_fnn.centers.copy(),
    'widths':  source_fnn.widths.copy(),
    'weights': source_fnn.weights.copy(),
}

target_fnn = FNN(n_inputs=n_inputs, n_rules=source_fnn.n_rules)
target_fnn.centers = source_fnn.centers.copy()
target_fnn.widths  = source_fnn.widths.copy()
target_fnn.weights = source_fnn.weights.copy()

rule_history, error_history = train_dk_sofnn(
    source_fnn, target_fnn,
    X_target_train, y_target_train_noisy,
    source_expert, source_avgs, source_params,
    N_T=N_T_DEFAULT, lr=TARGET_LR,
)

## 5. Evaluation (Eqs. 42-44)

In [ ]:
y_test_pred = target_fnn.predict(X_target_test).reshape(-1, 1)
rmse, smape, mase = calculate_metrics(y_target_test, y_test_pred)

print(f"Final rule count: {target_fnn.n_rules}")
print(f"RMSE  (Eq. 42): {rmse:.6f}")
print(f"sMAPE (Eq. 43): {smape:.6f}")
print(f"MASE  (Eq. 44): {mase:.6f}")

y_test_real = scaler_y.inverse_transform(y_target_test)
y_pred_real = scaler_y.inverse_transform(y_test_pred)
rmse_real = np.sqrt(np.mean((y_test_real - y_pred_real) ** 2))
print(f"RMSE (original scale): {rmse_real:.4f}")

print_rule_base(target_fnn, feature_names, scaler_X, scaler_y)
print("\nDK-SOFNN pipeline complete.")